In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y p7zip-full
!7z x "/content/drive/MyDrive/Colab_Notebooks/dataset/physionet.zip" -o"/content/data"


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,227 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

In [ ]:
from src.mask_generation import process_all_parallel
from src.detect_bad_masks import get_bad_masks_parallel,regenerate_missing


train_path="/content/data/train"

process_all_parallel(train_path,workers=12,print_overlay=False, thickness=4)
get_bad_masks_parallel(train_path,delete=True,workers=12)
regenerate_missing(train_path)

Processing ECGs: 100%|██████████| 977/977 [07:35<00:00,  2.15ecg/s]

DONE | ok=977 bad=0 total=977



Scanning npz files: 100%|██████████| 977/977 [00:13<00:00, 72.25it/s]


Total npz: 977
Bad npz: 0
The corrupted masks will be deleted? True
Missing masks: 0


Regenerating: 0it [00:00, ?it/s]


In [1]:
!pip -q install timm==0.9.2 segmentation-models-pytorch==0.3.3 unzip kaggle


In [2]:
import os, glob, random
import numpy as np
from PIL import Image
import cv2
import os, glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import segmentation_models_pytorch as smp

SEED = 42
random.seed(SEED);
np.random.seed(SEED);
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_PATH="/content/data"
TRAIN_ROOT = f"{BASE_PATH}/train"
DEBUG_DIR = "debug_train_evolution_stageA"
dataset_cache_dir = f"{BASE_PATH}/cache_ecg_stageA"

H_T = 864
W_T = 1120 # half-res of (1700,2200)
K=13 # Stage B out_channels

BATCH_SIZE = 24
NUM_WORKERS = 6

LR=2e-4
WEIGHT_DECAY=1e-4
EPOCHS = 25

In [3]:
import os, random
from src.PairedScanifyDataset import PairedScanifyDataset

all_ids = sorted([d for d in os.listdir(TRAIN_ROOT) if os.path.isdir(os.path.join(TRAIN_ROOT, d))])
random.shuffle(all_ids)

n = len(all_ids)
n_val = max(200, int(0.05 * n))
val_ids = all_ids[:n_val]
train_ids = all_ids[n_val:]

train_ds = PairedScanifyDataset(TRAIN_ROOT, train_ids, p_identity=0.10, cache_dir=dataset_cache_dir)
val_ds   = PairedScanifyDataset(TRAIN_ROOT, val_ids,   p_identity=0.10, cache_dir=dataset_cache_dir)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

In [4]:


# Stage A (student): hard -> clean image
stageA = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=1,
    classes=1,
    activation=None,
).to(DEVICE)

# Stage B (teacher): clean image -> masks (your existing trained model)
stageB = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,   # because you'll load your checkpoint weights
    in_channels=1,
    classes=K,
    activation=None,
).to(DEVICE)


# load your trained Stage A checkpoint
ckpt_path_stage_a = "/content/models/best_stageA_scanifier.pt"  # change if needed
state_a = torch.load(ckpt_path_stage_a, map_location=DEVICE)
stageA.load_state_dict(state_a['model'], strict=True)


# load your trained Stage B checkpoint
ckpt_path_stage_b = "/content/models/best_unet_resnet34_halfres_all_types.pt"  # change if needed
state_b = torch.load(ckpt_path_stage_b, map_location=DEVICE)
stageB.load_state_dict(state_b['model'], strict=True)



# freeze teacher
stageB.eval()
for p in stageB.parameters():
    p.requires_grad = False

# Only at the first training (optional but good) initialize Stage A encoder from Stage B encoder (COPIES values, no shared refs)
#####stageA.encoder.load_state_dict(stageB.encoder.state_dict(), strict=True)
for p in stageA.encoder.parameters():
    p.requires_grad = True

stageA.encoder.train()
for m in stageA.encoder.modules():
    if isinstance(m, torch.nn.BatchNorm2d):
        m.eval()  # stops running_mean/var updates

def freeze_bn_stats(module):
    if isinstance(module, torch.nn.BatchNorm2d):
        module.eval()


In [5]:
enc_lr = LR * 0.1
dec_lr = LR

opt = torch.optim.AdamW([
    {"params": stageA.encoder.parameters(), "lr": enc_lr},
    {"params": stageA.decoder.parameters(), "lr": dec_lr},
    {"params": stageA.segmentation_head.parameters(), "lr": dec_lr},
], weight_decay=WEIGHT_DECAY)


In [6]:
from tqdm.auto import tqdm
import torch
from torch.amp import autocast, GradScaler

from src.loss_function import (
    make_ink_weight,
    weighted_charbonnier,
    teacher_consistency_loss_weighted,
    grad_loss_weighted,
)

def _ensure_b1hw(x):
    # Accept [B,H,W] or [B,1,H,W]
    if x.ndim == 3:
        return x[:, None, :, :]
    return x

@torch.no_grad()
def ink_mean_stageB(stageB, x):
    """
    x: [B,1,H,W] in [0,1]
    returns scalar float = mean over batch+pixels of max_k(sigmoid(logits))
    """
    p = torch.sigmoid(stageB(x))          # [B,K,H,W]
    ink = p.max(dim=1).values.mean()      # scalar
    return float(ink)

def run_epoch_stageA(
    stageA,
    stageB,
    loader,
    opt=None,
    train=True,
    scaler=None,
    log_every=50,
    # ink weight map params
    ink_thr=0.15,
    ink_dilate=13,
    ink_alpha=30.0,
    ink_border=16,
    # loss weights
    w_rec=1.0,
    w_teach=1.0,
    w_grad=0.02,
    w_id=0.1,
    teach_beta=20.0,
    # extra debug
    print_ink_every=50,   # set None/0 to disable
):
    """
    Stage A epoch runner with tqdm, AMP, and ink diagnostics.

    loader yields: (src, tgt, is_id)
      src/tgt: [B,1,H,W] or [B,H,W] in [0,1]
      is_id: boolean or 0/1
    """
    if train and opt is None:
        raise ValueError("opt must be provided when train=True")

    stageA.train(train)
    if train:
        stageA.apply(freeze_bn_stats)   # freeze BN running stats
    stageB.eval()                       # teacher always eval

    if scaler is None:
        scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))

    tot, n = 0.0, 0
    pbar = tqdm(loader, desc="train" if train else "val", leave=False)

    for step, batch in enumerate(pbar, start=1):
        if batch is None:
            continue

        src, tgt, is_id = batch
        src = _ensure_b1hw(src.to(DEVICE, non_blocking=True))
        tgt = _ensure_b1hw(tgt.to(DEVICE, non_blocking=True))
        is_id = is_id.to(DEVICE, non_blocking=True).bool()

        if train:
            opt.zero_grad(set_to_none=True)

        grad_ctx = torch.enable_grad() if train else torch.no_grad()
        with grad_ctx:
            # Build waveform-focused weight map in fp32 (outside autocast)
            wmap = make_ink_weight(
                stageB, tgt,
                thr=ink_thr,
                dilate=ink_dilate,
                alpha=ink_alpha,
                border=ink_border
            )

            with autocast("cuda", enabled=(DEVICE == "cuda")):
                pred = torch.sigmoid(stageA(src)).clamp(0, 1)

                l_rec   = weighted_charbonnier(pred, tgt, wmap)
                l_teach = teacher_consistency_loss_weighted(stageB, pred, tgt, beta=teach_beta)
                l_grad  = grad_loss_weighted(pred, tgt, wmap)

                if is_id.any().item():
                    l_id = weighted_charbonnier(pred[is_id], tgt[is_id], wmap[is_id])
                else:
                    l_id = torch.tensor(0.0, device=DEVICE)

                loss = w_rec*l_rec + w_teach*l_teach + w_grad*l_grad + w_id*l_id

        if train:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        tot += float(loss.item())
        n += 1
        avg = tot / max(n, 1)

        # Optional ink diagnostics every N steps
        if print_ink_every and (step % print_ink_every == 0):
            with torch.no_grad():
                # teacher response on clean target
                ink_tgt = ink_mean_stageB(stageB, tgt)
                # teacher response on StageA output
                ink_pred = ink_mean_stageB(stageB, pred.detach())
            tqdm.write(f"[{'train' if train else 'val'}] ink_mean tgt={ink_tgt:.6f} | pred={ink_pred:.6f}")

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{avg:.4f}",
            rec=f"{l_rec.item():.4f}",
            grad=f"{l_grad.item():.4f}",
            teach=f"{l_teach.item():.4f}",
            id=f"{l_id.item():.4f}",
        )

        if log_every and (step % log_every == 0):
            tqdm.write(
                f"[{'train' if train else 'val'}] step {step}/{len(loader)}  "
                f"loss={loss.item():.4f}  avg={avg:.4f}  "
                f"rec={l_rec.item():.4f}  grad={l_grad.item():.4f}  "
                f"teach={l_teach.item():.4f}  id={l_id.item():.4f}"
            )

    return tot / max(n, 1), scaler


In [7]:
import os
import torch
from torch.amp import GradScaler

# --- Diagnostics ---
@torch.no_grad()
def ink_mean(stageB, x):
    """
    x: [B,1,H,W] in [0,1]
    returns scalar float
    """
    p = torch.sigmoid(stageB(x))                # [B,K,H,W]
    return float(p.max(dim=1).values.mean())    # mean of "ink confidence"

@torch.no_grad()
def debug_blank_check(stageA, debug_src, device, white_thr=0.98, white_frac=0.995,
                      std_thr=0.02, range_thr=0.10):
    """
    Returns (is_blank, dbg_pred)
    Blank if:
      - too many pixels are very white, OR
      - low std (flat), OR
      - low dynamic range
    """
    src = debug_src.to(device)
    if src.ndim == 3:
        src = src[:, None, :, :]
    pred = torch.sigmoid(stageA(src)).clamp(0,1)

    frac_white = float((pred > white_thr).float().mean())
    std = float(pred.std())
    rng = float((pred.max() - pred.min()))

    is_blank = (frac_white > white_frac) or (std < std_thr) or (rng < range_thr)
    return is_blank, pred


debug_loader = DataLoader(val_loader.dataset, batch_size=1, shuffle=False, num_workers=0)

debug_src = debug_tgt = debug_is_id = None
for s, t, is_id in debug_loader:
    if not is_id.item():
        debug_src, debug_tgt, debug_is_id = s.contiguous(), t.contiguous(), is_id
        break


assert debug_src is not None, "Could not sample a non-identity debug pair. Reduce p_identity or adjust dataset."


In [ ]:


# --- Training loop ---
from src.train import save_pred_debug_epoch_stageA

best_val = float("inf")
scaler = GradScaler("cuda", enabled=(DEVICE=="cuda"))

for ep in range(1, EPOCHS + 1):
    print(f"\n=== Epoch {ep}/{EPOCHS} ===")

    tr, scaler = run_epoch_stageA(
        stageA, stageB, train_loader,
        opt=opt, train=True,
        scaler=scaler,
        log_every=50,
    )

    va, _ = run_epoch_stageA(
        stageA, stageB, val_loader,
        opt=None, train=False,
        scaler=None,
        log_every=0,
    )

    print(f"epoch {ep}/{EPOCHS} | train {tr:.4f} | val {va:.4f}")

    # ---- Ink diagnostics on debug batch (MOST IMPORTANT) ----
    with torch.no_grad():
        ds = debug_src.to(DEVICE)
        dt = debug_tgt.to(DEVICE)
        if ds.ndim == 3: ds = ds[:, None, :, :]
        if dt.ndim == 3: dt = dt[:, None, :, :]

        pred_dbg = torch.sigmoid(stageA(ds)).clamp(0,1)
        ink_tgt = ink_mean(stageB, dt)
        ink_pred = ink_mean(stageB, pred_dbg)

    print(f"ink_mean: tgt={ink_tgt:.6f} | pred={ink_pred:.6f}")

    # ---- Save debug snapshot ----
    snap = save_pred_debug_epoch_stageA(
        stageA, stageB,
        debug_src, debug_tgt,
        epoch=ep,
        out_dir=DEBUG_DIR,
        ch_idxs=(1, 12),
        thr=0.3,
        device=DEVICE,
        show_teacher=True,
    )
    print("saved debug snapshot:", snap)

    # ---- Blank/flat collapse check ----
    is_blank, dbg_pred = debug_blank_check(stageA, debug_src, DEVICE)

    if is_blank and ep >= 5:
        print(
            f"⚠️ StageA output looks blank/flat at epoch {ep}. "
            f"Consider: (1) stronger teacher-weighting beta, "
            f"(2) lower grad weight, (3) crop training."
        )
        # Optional: early stop to save GPU money
        # break

    # ---- Save best checkpoint ----
    if va < best_val:
        best_val = va
        torch.save({"model": stageA.state_dict(), "val_loss": va, "epoch": ep},
                   "best_stageA_scanifier.pt")
        print("saved best checkpoint")



=== Epoch 1/25 ===


train:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
with torch.no_grad():
    tp = torch.sigmoid(stageB(debug_tgt.to(DEVICE)))
    sp = torch.sigmoid(stageB(torch.sigmoid(stageA(debug_src.to(DEVICE))).clamp(0,1)))
    print("teacher ink mean:", float(tp.max(1).values.mean()),
          "| student ink mean:", float(sp.max(1).values.mean()))


In [ ]:
with torch.no_grad():
    # Stage B response to TRUE clean target
    tb = torch.sigmoid(stageB(debug_tgt)).max(dim=1).values.mean().item()

    # Stage B response to Stage A output
    pred = torch.sigmoid(stageA(debug_src)).clamp(0,1)
    sb = torch.sigmoid(stageB(pred)).max(dim=1).values.mean().item()

print("B on tgt (ink mean):", tb)
print("B on pred (ink mean):", sb)
